In [1]:
import sys; sys.path.append("..")
import json, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from scipy.stats import loguniform, randint, uniform
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_validate
from xgboost import XGBClassifier
from src.features import get_splits, make_preprocessor
from src.evaluation import make_cv

X_train, X_test, y_train, y_test = get_splits("../data/raw/telco_churn.csv")

In [2]:
pipe = Pipeline([("pre", make_preprocessor()),
                 ("clf", XGBClassifier(eval_metric="logloss", tree_method="hist",
                                       random_state=42, n_jobs=1))])
space = {"clf__n_estimators": randint(100, 500),
         "clf__max_depth": randint(1, 5),
         "clf__learning_rate": loguniform(0.01, 0.2),
         "clf__subsample": uniform(0.6, 0.4),
         "clf__colsample_bytree": uniform(0.5, 0.5),
         "clf__min_child_weight": randint(1, 20),
         "clf__reg_lambda": loguniform(0.1, 20)}

search = RandomizedSearchCV(pipe, space, n_iter=40,
                            scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
                            refit="roc_auc", cv=make_cv(), n_jobs=-1, random_state=42)
search.fit(X_train, y_train)

best_params = {k.replace("clf__", ""): (v.item() if hasattr(v, "item") else v)
               for k, v in search.best_params_.items()}
print("best CV AUC: %.4f" % search.best_score_)
print(best_params)

res = pd.DataFrame(search.cv_results_)
print("top 5 AUCs:", res["mean_test_roc_auc"].nlargest(5).round(4).tolist())
print("median AUC over all 40:", round(res["mean_test_roc_auc"].median(), 4))

json.dump(best_params, open("../models/xgb_best_params.json", "w"), indent=2)

best CV AUC: 0.8506
{'colsample_bytree': 0.5127095633720475, 'learning_rate': 0.013815607382950859, 'max_depth': 3, 'min_child_weight': 7, 'n_estimators': 340, 'reg_lambda': 0.5288622176491503, 'subsample': 0.8034282764658811}
top 5 AUCs: [0.8506, 0.8505, 0.8504, 0.8504, 0.8504]
median AUC over all 40: 0.8472


In [3]:
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=7)
models = {
    "LogReg C=0.1": LogisticRegression(C=0.1, max_iter=2000),
    "RF depth=8": RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5,
                                         n_jobs=1, random_state=42),
    "XGB tuned": XGBClassifier(**best_params, eval_metric="logloss",
                               tree_method="hist", random_state=42, n_jobs=1),
}
scores = {}
for name, clf in models.items():
    p = Pipeline([("pre", make_preprocessor()), ("clf", clf)])
    s = cross_validate(p, X_train, y_train, cv=rcv, n_jobs=-1,
                       scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"})
    scores[name] = s
    print("%-14s ROC-AUC %.4f ± %.4f | PR-AUC %.4f ± %.4f" % (
        name, s["test_roc_auc"].mean(), s["test_roc_auc"].std(),
        s["test_pr_auc"].mean(), s["test_pr_auc"].std()))

for a in ["XGB tuned", "RF depth=8"]:
    d = scores[a]["test_roc_auc"] - scores["LogReg C=0.1"]["test_roc_auc"]
    print("%s minus LogReg: mean %+.4f, wins %d/15 folds" % (a, d.mean(), (d > 0).sum()))

LogReg C=0.1   ROC-AUC 0.8465 ± 0.0144 | PR-AUC 0.6624 ± 0.0282
RF depth=8     ROC-AUC 0.8460 ± 0.0140 | PR-AUC 0.6623 ± 0.0286
XGB tuned      ROC-AUC 0.8495 ± 0.0137 | PR-AUC 0.6705 ± 0.0269
XGB tuned minus LogReg: mean +0.0029, wins 12/15 folds
RF depth=8 minus LogReg: mean -0.0005, wins 5/15 folds
